# 국토교통부 건축물대장 정보로 건물 보완

In [16]:
import pandas as pd

cols = ['대지위치', '법정동코드명', '주용도코드명', '기타용도내용',
        '사용승인일자', '대지면적', '세대수']

df = pd.read_csv('../data/서울시 건축물대장 표제부.csv',
                 encoding='cp949',
                 encoding_errors='ignore',
                 on_bad_lines='skip',
                 usecols=cols)
print('행 개수 : ', len(df))
print('열 개수 : ', len(df.columns))
print(df.head())

행 개수 :  593048
열 개수 :  7
                   대지위치 법정동코드명   대지면적     주용도코드명   기타용도내용   세대수      사용승인일자
0  서울특별시 강서구 방화동 246-66    방화동  274.0       단독주택  주택 및 창고   NaN  1989-06-30
1  서울특별시 마포구 아현동 751-11    아현동    0.0       공동주택     연립주택   6.0  1984-08-06
2   서울특별시 양천구 신월동 987-1    신월동    0.0       공동주택      아파트  96.0  1988-04-30
3   서울특별시 양천구 신정동 87-39    신정동    0.0  제1종근린생활시설   근린생활시설   0.0  1990-12-29
4  서울특별시 강서구 화곡동 102-29    화곡동  172.9       단독주택     단독주택   0.0  1990-08-18


In [17]:
# 연립 + 다세대 필터링
df = df[df['기타용도내용'].str.contains('연립|다세대', na=False)]

# 건축연도 추출 및 2022 이하 필터
df['건축연도'] = pd.to_numeric(
    df['사용승인일자'].astype(str).str[:4], errors='coerce')
df = df[df['건축연도'].notna() & (df['건축연도'] <= 2022)]

print(df.shape)
print(df['기타용도내용'].value_counts().head(10))

(79832, 8)
기타용도내용
다세대주택                32450
공동주택(다세대주택)           8364
연립주택                  5530
도시형생활주택(단지형다세대)       2278
다세대주택(8세대)            1740
공동주택,다세대주택            1188
도시형생활주택(단지형다세대주택)     1131
공동주택(다세대)              983
다세대주택, 근린생활시설          908
다세대주택(9세대)             871
Name: count, dtype: int64


In [18]:
# 결측값 확인
print("=== 결측값 ===")
print(df.isnull().sum())

# 대지면적 0인 행 확인
print("\n=== 대지면적 0 또는 결측 ===")
print(f"대지면적 0: {(df['대지면적'] == 0).sum()}")
print(f"대지면적 결측: {df['대지면적'].isna().sum()}")

# 세대수 확인
print("\n=== 세대수 0 또는 결측 ===")
print(f"세대수 0: {(df['세대수'] == 0).sum()}")
print(f"세대수 결측: {df['세대수'].isna().sum()}")

# 사용승인일자 결측
print("\n=== 사용승인일자 결측 ===")
print(f"결측: {df['사용승인일자'].isna().sum()}")

# 건축연도 분포
print("\n=== 건축연도 분포 ===")
print(df['건축연도'].describe())
print(df['건축연도'].value_counts().sort_index().tail(10))

=== 결측값 ===
대지위치         0
법정동코드명       0
대지면적      2216
주용도코드명       0
기타용도내용       0
세대수         42
사용승인일자       0
건축연도         0
dtype: int64

=== 대지면적 0 또는 결측 ===
대지면적 0: 11431
대지면적 결측: 2216

=== 세대수 0 또는 결측 ===
세대수 0: 321
세대수 결측: 42

=== 사용승인일자 결측 ===
결측: 0

=== 건축연도 분포 ===
count    79832.000000
mean      2002.857939
std         11.067144
min       1940.000000
25%       1994.000000
50%       2002.000000
75%       2013.000000
max       2022.000000
Name: 건축연도, dtype: float64
건축연도
2013.0    1952
2014.0    2535
2015.0    2980
2016.0    3667
2017.0    2458
2018.0    1847
2019.0    1604
2020.0    1248
2021.0    1311
2022.0    1179
Name: count, dtype: int64


In [19]:
# 노후도는 전체 79,832건 다 사용 가능
# 집적도는 대지면적 > 0 인 것만 사용
df_density = df[df['대지면적'] > 0]  # 66,185건
df_age = df.copy()                   # 79,832건

print(f"노후도 계산 대상: {len(df_age)}건")
print(f"집적도 계산 대상: {len(df_density)}건")

# 행정동별 노후도
df_age['노후'] = (2022 - df_age['건축연도']) >= 30

# 행정동별 집적도 (좌표 매핑 후)
# unit_density = 세대수 합계 / 대지면적 합계

노후도 계산 대상: 79832건
집적도 계산 대상: 66185건


In [22]:
df.groupby('법정동코드명').size().sort_values(ascending=False)

법정동코드명
화곡동       4755
신월동       2107
상도동       1558
봉천동       1477
목동        1451
          ... 
무학동          1
합동           1
을지로6가        1
동소문동3가       1
원지동          1
Length: 338, dtype: int64

In [24]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

small = df.groupby('법정동코드명').filter(lambda x: len(x) <= 3)
print(small[['법정동코드명', '대지위치', '기타용도내용']])

        법정동코드명                     대지위치                                기타용도내용
2982       율현동     서울특별시 강남구 율현동 309-22                                 다세대주택
11398     종로6가     서울특별시 종로구 종로6가 201-1                                 다세대주택
37904      운니동       서울특별시 종로구 운니동 45-3                          다세대(도시형생활주택)
41905      충신동      서울특별시 종로구 충신동 25-18                                 다세대주택
45917       화동          서울특별시 종로구 화동 78                                  연립주택
45979      산천동       서울특별시 용산구 산천동 1-29                                 다세대주택
49402      남영동        서울특별시 용산구 남영동 104                          근린생활시설/다세대주택
49457      통의동        서울특별시 종로구 통의동 9-1                            공동주택,다세대주택
53665      원지동      서울특별시 서초구 원지동 399-1                                 다세대주택
63157    문래동1가    서울특별시 영등포구 문래동1가 22-7                              다세대,오피스텔
83051      율현동     서울특별시 강남구 율현동 196-86                           공동주택(다세대주택)
102054     원남동     서울특별시 종로구 원남동 243-10                         

In [26]:
cols = ['대지위치', '법정동코드명', '주용도코드명', '기타용도내용',
        '사용승인일자', '대지면적', '세대수']

df_원본 = pd.read_csv('../data/서울시 건축물대장 표제부.csv',
                 encoding='cp949',
                 encoding_errors='ignore',
                 on_bad_lines='skip',
                 usecols=cols)

In [27]:
# 건축물대장에서 주용도코드명 전체 분포 확인
print(df_원본['주용도코드명'].value_counts())
print(df_원본['기타용도내용'].str.contains('연립|다세대', na=False).sum())

# 혹시 공동주택 전체로 넓히면?
print(df_원본[df_원본['주용도코드명'] == '공동주택'].shape)

주용도코드명
단독주택          290155
공동주택          141801
제2종근린생활시설      63319
제1종근린생활시설      57877
업무시설           10288
교육연구시설          8256
노유자시설           3250
종교시설            2954
공장              2356
숙박시설            1988
자동차관련시설         1888
창고시설            1583
문화및집회시설         1421
판매시설            1225
근린생활시설           959
의료시설             799
위험물저장및처리시설       668
운동시설             355
교정및군사시설          290
운수시설             268
관광휴게시설           182
교육연구및복지시설        170
동물및식물관련시설        170
위락시설             167
분뇨.쓰레기처리시설       142
방송통신시설            94
국방,군사시설           93
자원순환관련시설          77
수련시설              73
묘지관련시설            47
판매및영업시설           41
발전시설              23
공공용시설             12
야영장시설             10
장례시설               6
다가구주택              2
가설건축물              1
교정시설               1
Name: count, dtype: int64
81200
(141801, 7)


In [28]:
# 공동주택 중 기타용도내용 분포 전체 확인
df_공동 = df_원본[df_원본['주용도코드명'] == '공동주택']
print(df_공동['기타용도내용'].value_counts().head(30))

기타용도내용
다세대주택                    32560
공동주택                     17757
아파트                       9854
공동주택(아파트)                 9420
공동주택(다세대주택)               8469
연립주택                      5416
경비실                       2754
도시형생활주택(단지형다세대)           2371
다세대주택(8세대)                1742
주거시설                      1457
지하주차장                     1198
도시형생활주택(단지형다세대주택)         1189
공동주택,다세대주택                1188
공동주택(다세대)                  983
다세대주택, 근린생활시설              915
다세대주택(9세대)                 873
도시형생활주택                    870
다세대주택(도시형생활주택/단지형다세대)      825
다세대주택(10세대)                738
다세대주택(6세대)                 713
주택                         659
공동주택(도시형생활주택)              600
다세대주택(7세대)                 582
다세대주택 및 근린생활시설             473
다세대주택,근린생활시설               456
다세대주택(4세대)                 411
경로당                        398
공동주택(연립주택)                 374
다세대주택(도시형생활주택)             370
다세대주택(5세대)                 334
Name: count, dtype: int64


In [29]:
print(df_공동['기타용도내용'].isna().sum())
print(df_공동['기타용도내용'].eq('').sum())

24
0


In [31]:
# "공동주택"으로만 표기된 것들의 실제 내용 샘플 확인
mask = df_원본['기타용도내용'] == '공동주택'
print(df_원본[mask][['대지위치', '기타용도내용', '세대수']].sample(20))

                          대지위치 기타용도내용    세대수
567511    서울특별시 강북구 번동 148-270   공동주택    6.0
468996     서울특별시 관악구 봉천동 674-5   공동주택    8.0
140024    서울특별시 송파구 방이동 187-15   공동주택    8.0
463445   서울특별시 관악구 신림동 254-134   공동주택   19.0
514254   서울특별시 서초구 서초동 1507-25   공동주택    8.0
216975  서울특별시 관악구 봉천동 1690-158   공동주택    7.0
199888    서울특별시 금천구 시흥동 492-12   공동주택    8.0
138373     서울특별시 구로구 개봉동 366-3   공동주택    8.0
576265      서울특별시 서초구 반포동 60-5   공동주택  150.0
310483  서울특별시 영등포구 대림동 1011-18   공동주택   10.0
307615    서울특별시 금천구 시흥동 854-29   공동주택    8.0
399406    서울특별시 송파구 오금동 114-28   공동주택   11.0
106707     서울특별시 용산구 원효로3가 143   공동주택    4.0
384625      서울특별시 광진구 중곡동 76-6   공동주택    9.0
186241   서울특별시 강북구 미아동 258-116   공동주택    6.0
207118    서울특별시 도봉구 도봉동 573-25   공동주택    6.0
39953     서울특별시 관악구 봉천동 860-53   공동주택    8.0
192121    서울특별시 강북구 미아동 258-51   공동주택    6.0
88648        서울특별시 동작구 상도동 431   공동주택   30.0
213830    서울특별시 중랑구 망우동 152-18   공동주택    6.0


In [33]:
mask = (
    df_원본['기타용도내용'].str.contains('연립|다세대', na=False) |
    (
        (df_원본['기타용도내용'] == '공동주택') & 
        (df_원본['세대수'] <= 50)
    )
)
df_filtered = df_원본[mask]
print(df_filtered.shape)

(97530, 7)
